# Reasoning Evaluation: Chain-of-Thought and Process Reward Models

Accuracy on math and logic benchmarks measures the final answer but ignores whether the model reasoned correctly to get there. A model can arrive at the right answer through flawed steps, and a model can have correct intermediate reasoning but make an arithmetic error at the end. Step-level evaluation separates these failure modes.

## Why Final-Answer Accuracy Is Insufficient

Two models might both achieve 70% on GSM8K (grade school math), but for very different reasons:
- Model A reasons correctly but makes arithmetic errors in the final computation
- Model B memorizes solution patterns without understanding, and fails on novel phrasings

For applications where reasoning process matters (tutoring, scientific analysis, code explanation), you need to verify the steps, not just the answer.

For RLHF training data, verifying correctness at the step level allows Process Reward Models (PRMs) to provide denser training signal than outcome-only rewards.

## Chain-of-Thought Evaluation

CoT evaluation requires:
1. **Step extraction**: parse the model's reasoning into discrete steps
2. **Step verification**: for each step, check if it follows from prior steps
3. **Error localization**: identify the first incorrect step

For math problems, step verification can be partially automated using symbolic evaluation (sympy, regex-based arithmetic checks). For natural language reasoning, step verification still requires LLM judges or human annotation.

In [ ]:
from dataclasses import dataclass
import re

@dataclass
class ReasoningEval:
    problem: str
    steps: list
    step_verdicts: list
    final_answer: str
    answer_correct: bool
    first_error_step: int

def parse_steps(cot_output: str) -> list:
    lines = cot_output.strip().split('\n')
    steps = []
    for line in lines:
        line = line.strip()
        if line and not line.startswith('#'):
            steps.append(line)
    return steps

def check_step(step: str, context: list) -> dict:
    # Check arithmetic expressions like '3 + 4 = 7'
    arith_match = re.search(r'(\d+(?:\.\d+)?)\s*([+\-*/])\s*(\d+(?:\.\d+)?)\s*=\s*(\d+(?:\.\d+)?)', step)
    if arith_match:
        a, op, b, claimed = arith_match.groups()
        a, b, claimed = float(a), float(b), float(claimed)
        ops = {'+': a+b, '-': a-b, '*': a*b, '/': a/b if b != 0 else None}
        expected = ops.get(op)
        correct = expected is not None and abs(expected - claimed) < 0.01
        return {'type': 'arithmetic', 'correct': correct,
                'expected': expected, 'claimed': claimed}
    return {'type': 'unknown', 'correct': None}

def eval_cot(problem: str, cot_output: str, correct_answer: str) -> ReasoningEval:
    steps = parse_steps(cot_output)
    verdicts = []
    first_error = -1
    for i, step in enumerate(steps):
        v = check_step(step, steps[:i])
        verdicts.append(v)
        if v['correct'] is False and first_error == -1:
            first_error = i
    # Extract final answer from last step
    final_match = re.search(r'(?:answer|result)[:\s]+([\d.]+)', steps[-1], re.I) if steps else None
    final_answer = final_match.group(1) if final_match else steps[-1] if steps else ''
    answer_correct = correct_answer.strip() in steps[-1] if steps else False
    return ReasoningEval(
        problem=problem,
        steps=steps,
        step_verdicts=verdicts,
        final_answer=final_answer,
        answer_correct=answer_correct,
        first_error_step=first_error
    )

problem = 'If a store has 12 apples and sells 5, then receives 8 more, how many apples are there?'
cot = (
    'Start with 12 apples.\n'
    'After selling 5: 12 - 5 = 7\n'
    'After receiving 8 more: 7 + 8 = 15\n'
    'The answer is 15'
)
result = eval_cot(problem, cot, '15')
print(f'Problem: {result.problem}')
print(f'Answer correct: {result.answer_correct}')
print(f'First error step: {result.first_error_step}')
print('Steps:')
for step, verdict in zip(result.steps, result.step_verdicts):
    status = 'OK' if verdict['correct'] else ('ERROR' if verdict['correct'] is False else '---')
    print(f'  [{status}] {step}')

## Process Reward Models

PRMs (Lightman et al. 2023, OpenAI) assign a reward to each step in a reasoning chain rather than only to the final answer. This enables:
- More informative RLHF training signal (each step is a reward signal)
- Better detection of lucky correct answers via flawed reasoning
- Targeted debugging of where models break down

Training PRMs requires human annotations at the step level — expensive but more informative than outcome labels. The Let's Verify Step by Step paper showed that PRM-selected solutions outperformed outcome-selected solutions on MATH benchmark.

In 2024-2025, step-level verification became central to OpenAI o1 and similar reasoning models — the internal reasoning chain is explicitly optimized to be verifiable at each step.

## Benchmark-Specific Reasoning Metrics

**GSM8K**: 8,500 grade school math word problems. Metric: exact match on final numerical answer. Success rate for top models: 95%+ by 2024.

**MATH** (Hendrycks et al. 2021): competition-level math problems across 5 difficulty levels. Much harder — GPT-4 reaches ~50% on the hardest level.

**ARC-Challenge**: science reasoning requiring multi-step inference. Unlike GSM8K, there is no clear intermediate verification strategy.

**LogiQA / FOLIO**: formal logical reasoning. Allows more systematic step verification since the steps correspond to formal inference rules.

The field is moving toward benchmarks where the reasoning trace is part of the evaluation, not just the answer.